# Visualize hydrogen bonds

Example of visualizing hydrogen bonds:

<video controls src="./assets/gluhut_hbonds.webm">

Load a recording (unbinding of a sugar from GluHUT) as an MDAnalsis universe:

In [1]:
from nanover.mdanalysis import universe_from_recording

universe = universe_from_recording("gluhut-unbinding.nanover.zip")

Use MDAnalysis to find all hydrogen bonds in the trajectory:

In [2]:
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA

hbond = HBA(
    universe=universe,
    donors_sel="resid 1 and name N* O*",
    hydrogens_sel="resid 1 and name H*",
    acceptors_sel="(resid 2) and (name O*)",
    d_a_cutoff=3.0,
    d_h_a_angle_cutoff=150.0,
    update_selections=True
)
hbond.run()

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\analysis\hydrogenbonds\hbond_analysis.py:779: UserWarning: No hydrogen bonds were found given angle of 150.0 between Donor, resid 1 and name N* O*, and Acceptor, (resid 2) and (name O*).
  warnings.warn(
C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\analysis\hydrogenbonds\hbond_analysis.py:747: UserWarning: No hydrogen bonds were found given d-a cutoff of 3.0 between Donor, resid 1 and name N* O*, and Acceptor, (resid 2) and (name O*).
  warnings.warn(


Process hydrogen bond results into listing of hbond per frame:

In [3]:
from typing import NamedTuple
from itertools import groupby

class HBond(NamedTuple):
    frame: int
    donor: int
    hydrogen: int
    acceptor: int
    distance: float
    angle: float

    @classmethod
    def from_row(cls, row):
        frame, donor, hydrogen, acceptor, distance, angle = row
        return cls(int(frame), int(donor), int(hydrogen), int(acceptor), distance, angle)

frame_to_hbonds = {
    frame: list(hbonds)
    for frame, hbonds in groupby(
        (HBond.from_row(row) for row in hbond.results.hbonds),
        lambda hbond: hbond.frame,
    )
}

Set up the server with playback of the universe:

In [4]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulation = UniverseSimulation.from_universe(universe, name="nanotube + methane")
simulation.playback_factor = 30
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: visualize structural metrics")
imd_runner.load(0)

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Import the jupyter utilities for drawing + interaction:

In [5]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

Define and start a simple FrameListener that renders the relevant hbonds when the frame is updated:

In [6]:
from nanover.trajectory import FrameData
from nanover.jupyter import FrameListener

class HBondVisuals(FrameListener):
    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        with utilities.objects:
            utilities.objects.clear()
            frame = simulation.prev_frame
            hbonds = frame_to_hbonds.get(frame, [])

            for i, hbond in enumerate(hbonds):
                a, b = full_frame.particle_positions[[hbond.hydrogen, hbond.acceptor]]
                utilities.objects.update_line(f"hbond.{i}", positions=[a, b])

visuals = HBondVisuals.from_runner(imd_runner)
visuals.start()

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt
